# 01 · MedQuAD data preparation

CIS 5300: Medical Question Intent Classification. The original XML comes from MedQuAD by Ben Abacha and Demner-Fushman (CC BY 4.0). This notebook fixes the dataset version, extracts questions and metadata, selects six disease-related intent classes, and freezes an approximately 80/10/10 split by disease group. Answers never enter the classifier.

Before running: install dependencies from the project root as described in the README. The student must confirm dataset approval and permission for AI assistance with the teaching team. This code was prepared with Codex assistance.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "config.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "config.json").exists(), "Open this notebook inside the project."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)
import pandas as pd


## Parse, clean, and freeze

The parser also supports legacy XML with lowercase tags. Disease scope follows the original annotations. Conflicting labels are quarantined, and identical questions with the same label are deduplicated. Existing frozen splits cannot be silently changed.

In [2]:
from src.prepare_data import prepare
questions = prepare()
questions[['question_id', 'question_text', 'label', 'focus', 'split']].head()

Verified 11274 XML files at 577bd37b96c02d1833b2c9eed2de9f96964e96cb


{
  "raw_questions": 47457,
  "retained_questions": 17845,
  "xml_files": 11274,
  "parsed_issues": {
    "no_questions": 6,
    "legacy_schema_supported": 5
  },
  "dispositions": {
    "retained": 17845,
    "non_disease_or_unknown_category": 15754,
    "outside_six_intents": 10698,
    "same_text_same_label_duplicate": 3148,
    "missing_focus": 12
  },
  "classes": {
    "Basic information": 5302,
    "Symptoms": 4039,
    "Treatment": 3221,
    "Causes": 2217,
    "Diagnosis and tests": 1891,
    "Prevention": 1175
  },
  "disease_groups": 6238,
  "splits": {
    "train": {
      "questions": 14158,
      "groups": 4990
    },
    "dev": {
      "questions": 1880,
      "groups": 624
    },
    "test": {
      "questions": 1807,
      "groups": 624
    }
  },
  "cross_split_disease_groups": 0,
  "cross_split_normalized_duplicate_questions": 0,
  "unresolved_cross_split_cui_count": 471,
  "grouping_claim": "available annotated focus/synonym grouping; not full medical concept isolat

,question_id,question_text,label,focus,split
0,10_MPlus_ADAM_QA/0000001.xml::0000001-1,What is (are) A guide to clinical trials for c...,Basic information,A guide to clinical trials for cancer,train
6,10_MPlus_ADAM_QA/0000003.xml::0000003-1,What is (are) A1C test ?,Basic information,A1C test,train
8,10_MPlus_ADAM_QA/0000004.xml::0000004-1,What is (are) Aarskog syndrome ?,Basic information,Aarskog syndrome,train
9,10_MPlus_ADAM_QA/0000004.xml::0000004-2,What causes Aarskog syndrome ?,Causes,Aarskog syndrome,train
10,10_MPlus_ADAM_QA/0000004.xml::0000004-3,What are the symptoms of Aarskog syndrome ?,Symptoms,Aarskog syndrome,train


In [3]:
pd.read_csv('results/class_counts.csv')

,label,questions,disease_groups,median_words
0,Basic information,5302,5060,7.0
1,Causes,2217,2185,5.0
2,Diagnosis and tests,1891,1870,6.0
3,Prevention,1175,1166,6.0
4,Symptoms,4039,3934,9.0
5,Treatment,3221,3100,8.0


In [4]:
pd.read_csv('results/split_class_counts.csv')

,split,Basic information,Symptoms,Causes,Diagnosis and tests,Treatment,Prevention
0,dev,543,405,238,212,352,130
1,test,539,401,221,194,330,122
2,train,4220,3233,1758,1485,2539,923


In [5]:
from IPython.display import Markdown, display
display(Markdown(Path('results/data_audit.md').read_text()))

# Data-stage audit

Generated from the real pinned XML; no classifier was evaluated at this stage.

- Parsed questions: 47457
- Retained questions: 17845
- Disease groups: 6238
- Split SHA-256: `6e67ba6b510efeea0a567fd0963842ca03297c19ba6d5fd17de959061b69fe66`

| Split | Questions | Groups |
|---|---:|---:|
| train | 14158 | 4990 |
| dev | 1880 | 624 |
| test | 1807 | 624 |

| Class | Questions |
|---|---:|
| Basic information | 5302 |
| Symptoms | 4039 |
| Causes | 2217 |
| Diagnosis and tests | 1891 |
| Treatment | 3221 |
| Prevention | 1175 |

Checks: unique IDs, six-class coverage in every split, disjoint disease groups and normalized questions passed.

Grouping uses normalized focus names and unambiguous explicitly annotated synonym-to-focus links, rejecting disjoint annotated CUI sets. Identical-question links couple split allocation before deduplication. Shared CUIs alone do not establish disease equivalence.

Unresolved CUI IDs spanning splits: 471. See unresolved_cross_split_cuis.csv; complete concept isolation is not claimed. MedQuAD Disease annotations are accepted as supplied and may include broad health topics. Clinical scope and synonym audit remain limitations.

Same-text conflicting labels are quarantined pending human review. See label_conflicts.csv and row_disposition.csv. Missing answers do not cause exclusion. Template replacement is used only for auditing.

Pending user confirmation: instructor dataset approval, cross-course reuse permission and permitted generative-AI assistance. No approval is inferred. Human-written/reviewed paraphrases have not been completed.


## Stage A checks

Check unique IDs, disjoint disease groups, disjoint normalized questions, and coverage of every class in each split. Grouping is limited by the completeness of synonym annotations. Shared CUIs alone do not establish disease equivalence; unresolved cross-split CUI overlap is recorded in the audit. Test labels are used here only to check split coverage; no classifier test scores are computed.

In [6]:
from src.prepare_data import validate_splits
validate_splits(questions)
print('Data checks passed; no classifier test scores computed.')

Data checks passed; no classifier test scores computed.
